## Implement a Random Forest model from scratch without using scikit-learn, manually constructing multiple decision trees, aggregating predictions, and evaluating performance using custom accuracy calculations.

In [1]:
import pandas as pd
import random
import math

In [2]:
df = pd.read_csv("heart.csv")

print(df.head())
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   63    1   3       145   233    1        0      150      0      2.3      0   
1   37    1   2       130   250    0        1      187      0      3.5      0   
2   41    0   1       130   204    0        0      172      0      1.4      2   
3   56    1   1       120   236    0        1      178      0      0.8      2   
4   57    0   0       120   354    0        1      163      1      0.6      2   

   ca  thal  target  
0   0     1       1  
1   0     2       1  
2   0     2       1  
3   0     2       1  
4   0     2       1  
Dataset shape: (303, 14)
Columns: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']


In [3]:
X = df.drop("target", axis=1).values.tolist()
y = df["target"].tolist()

print("Total input rows:", len(X))
print("Total features:", len(X[0]))
print("Target values:", set(y))

Total input rows: 303
Total features: 13
Target values: {0, 1}


In [4]:
random.seed(42)

data = list(zip(X, y))
random.shuffle(data)

split_point = int(len(data) * 0.8)

train_data = data[:split_point]
test_data = data[split_point:]

X_train = [row[0] for row in train_data]
y_train = [row[1] for row in train_data]

X_test = [row[0] for row in test_data]
y_test = [row[1] for row in test_data]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 242
Testing rows: 61


In [5]:
def calculate_gini(labels):
    if len(labels) == 0:
        return 0

    counts = {}

    for label in labels:
        if label not in counts:
            counts[label] = 0
        counts[label] += 1

    gini = 1

    for count in counts.values():
        probability = count / len(labels)
        gini = gini - probability ** 2

    return gini

In [6]:
def find_best_split(X, y, feature_count):
    best_score = float("inf")
    best_split = None

    feature_ids = list(range(len(X[0])))
    random.shuffle(feature_ids)

    feature_ids = feature_ids[:feature_count]

    for feature in feature_ids:
        values = list(set(row[feature] for row in X))
        values.sort()

        if len(values) > 10:
            step = max(1, len(values) // 10)
            values = values[::step]

        for i in range(len(values) - 1):
            threshold = (values[i] + values[i + 1]) / 2

            left_y = []
            right_y = []

            for row, label in zip(X, y):
                if row[feature] <= threshold:
                    left_y.append(label)
                else:
                    right_y.append(label)

            if len(left_y) == 0 or len(right_y) == 0:
                continue

            score = (
                len(left_y) / len(y) * calculate_gini(left_y)
                + len(right_y) / len(y) * calculate_gini(right_y)
            )

            if score < best_score:
                best_score = score
                best_split = (feature, threshold)

    return best_split

In [7]:

def build_tree(X, y, depth=0, max_depth=4):
    counts = {}

    for label in y:
        counts[label] = counts.get(label, 0) + 1

    majority_class = max(counts, key=counts.get)

    if len(counts) == 1 or depth >= max_depth or len(y) < 5:
        return majority_class

    feature_count = max(1, int(math.sqrt(len(X[0]))))

    split = find_best_split(X, y, feature_count)

    if split is None:
        return majority_class

    feature, threshold = split

    left_X = []
    left_y = []
    right_X = []
    right_y = []

    for row, label in zip(X, y):
        if row[feature] <= threshold:
            left_X.append(row)
            left_y.append(label)
        else:
            right_X.append(row)
            right_y.append(label)

    if len(left_y) == 0 or len(right_y) == 0:
        return majority_class

    left_tree = build_tree(left_X, left_y, depth + 1, max_depth)
    right_tree = build_tree(right_X, right_y, depth + 1, max_depth)

    return (feature, threshold, left_tree, right_tree)


In [8]:

def predict_tree(tree, row):
    if not isinstance(tree, tuple):
        return tree

    feature, threshold, left_tree, right_tree = tree

    if row[feature] <= threshold:
        return predict_tree(left_tree, row)
    else:
        return predict_tree(right_tree, row)


In [9]:

def train_forest(X, y, tree_count=7):
    forest = []

    for i in range(tree_count):
        sample_X = []
        sample_y = []

        for j in range(len(X)):
            index = random.randint(0, len(X) - 1)
            sample_X.append(X[index])
            sample_y.append(y[index])

        tree = build_tree(sample_X, sample_y)
        forest.append(tree)

    return forest


forest = train_forest(X_train, y_train, tree_count=7)

print("Total decision trees:", len(forest))
print("Random Forest training completed!")


Total decision trees: 7
Random Forest training completed!


In [10]:

def predict_forest(forest, row):
    votes = []

    for tree in forest:
        prediction = predict_tree(tree, row)
        votes.append(prediction)

    counts = {}

    for vote in votes:
        counts[vote] = counts.get(vote, 0) + 1

    return max(counts, key=counts.get)


predictions = []

for row in X_test:
    result = predict_forest(forest, row)
    predictions.append(result)

print("Actual values:    ", y_test[:10])
print("Predicted values: ", predictions[:10])


Actual values:     [1, 1, 0, 1, 1, 0, 1, 0, 1, 1]
Predicted values:  [1, 1, 0, 0, 1, 0, 0, 0, 0, 1]


In [11]:

correct = 0

for actual, predicted in zip(y_test, predictions):
    if actual == predicted:
        correct += 1

accuracy = (correct / len(y_test)) * 100

print("Correct predictions:", correct)
print("Total test records:", len(y_test))
print("Random Forest Accuracy:", round(accuracy, 2), "%")


Correct predictions: 49
Total test records: 61
Random Forest Accuracy: 80.33 %


In [12]:

patient = X_test[0]

result = predict_forest(forest, patient)

print("Patient features:", patient)
print("Predicted target:", result)
print("Actual target:", y_test[0])


Patient features: [41.0, 1.0, 1.0, 120.0, 157.0, 0.0, 1.0, 182.0, 0.0, 0.0, 2.0, 0.0, 2.0]
Predicted target: 1
Actual target: 1
